# TERF — ray tracing LOS/NLOS (stazione permanente)

Pipeline Colab per la stazione **TERF** (`TERF00CYP`, Cyprus).

**Input:** RINEX OBS `TERF00CYP_R_*_MO.rnx` (upload manuale o copia da repo).

**Output** in `/content/terf_work/results/`:
- mesh OSM (`terf_osm_triangles.npy`, `terf_osm.glb`)
- `reference_YYYYDOY.csv` (ECEF fisso)
- `terf_los_labels_YYYYDOY_gc.csv` (G+C)
- `terf_pipeline_summary.json`

**Runtime:** GPU (T4) + build CUDA `_bvh`.

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone repo (sostituisci URL con il tuo fork/branch)
import os
REPO = "/content/gnss_gpu"
if not os.path.isdir(REPO):
    !git clone https://github.com/YOUR_USER/gnss_gpu.git {REPO}
%cd {REPO}

In [ ]:
# 3) Dipendenze + build CUDA (T4 = sm_75)
!pip install -q numpy matplotlib folium branca pyproj rasterio requests scipy
!apt-get -qq install -y cmake
!mkdir -p build
%cd build
!cmake .. -DCMAKE_CUDA_ARCHITECTURES=75
!make -j$(nproc)
%cd ..

In [ ]:
# 4) Test import estensioni
import os, sys
os.environ["PYTHONPATH"] = "python:build"
sys.path[:0] = ["python", "build"]
import gnss_gpu._bvh
print("_bvh OK")

In [ ]:
# 5a) Opzione A — copia RINEX dal repo (se hai committato experiments/data/TERF)
import os
os.environ["PYTHONPATH"] = "python:build"
!mkdir -p /content/terf_work/data
!python experiments/run_terf_permanent_station_colab.py --phase setup --work-dir /content/terf_work

In [ ]:
# 5b) Opzione B — upload manuale dei file TERF*.rnx
from google.colab import files
import shutil
from pathlib import Path

dest = Path("/content/terf_work/data")
dest.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # seleziona TERF00CYP_R_*_MO.rnx
for name, data in uploaded.items():
    (dest / name).write_bytes(data)
    print("saved", dest / name)

In [ ]:
# 6) Pipeline completa: mesh OSM → BRDC → labels → summary
import os
os.environ["PYTHONPATH"] = "python:build"

# Prima esecuzione veloce: --epoch-step 30 (1 epoca/min)
# Produzione: --epoch-step 1
!python experiments/run_terf_permanent_station_colab.py \
  --work-dir /content/terf_work \
  --phase all \
  --systems G,C \
  --epoch-step 1

In [ ]:
# 7) Riepilogo risultati
from IPython.display import JSON, display
import json
from pathlib import Path

summary_path = Path("/content/terf_work/results/terf_pipeline_summary.json")
if summary_path.exists():
    display(JSON(json.loads(summary_path.read_text())))
else:
    print("Summary not found:", summary_path)

for p in sorted(Path("/content/terf_work/results").glob("terf_los_labels_*_gc.csv")):
    print(p.name, p.stat().st_size, "bytes")

In [ ]:
# 8) Scarica zip risultati
!zip -r /content/terf_results.zip /content/terf_work/results
from google.colab import files
files.download("/content/terf_results.zip")

## Fasi singole (debug)

```bash
python experiments/run_terf_permanent_station_colab.py --phase mesh --work-dir /content/terf_work
python experiments/run_terf_permanent_station_colab.py --phase nav --work-dir /content/terf_work
python experiments/run_terf_permanent_station_colab.py --phase labels --work-dir /content/terf_work --epoch-step 30
python experiments/run_terf_permanent_station_colab.py --phase summary --work-dir /content/terf_work
```

Test rapido (pochi minuti): `--epoch-step 30 --max-epochs 96`.